The purpose of this template file is to do pre-processing and feature selection.

In [1]:
!pip install braindecode[hub]==1.8.1
import braindecode

/usr/local/lib/python3.13/dist-packages/braindecode/models/eegpt.py:497: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = mne.channels.make_standard_montage("standard_1020")
/usr/local/lib/python3.13/dist-packages/braindecode/models/eegpt.py:1452: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = make_standard_montage("standard_1020")


In [2]:
import os
import importlib
import sys
from pathlib import Path
import json

from tqdm import tqdm

import numpy as np
import matplotlib
from matplotlib import pyplot as plt

from sklearn.manifold import TSNE

import h5py
import cv2

In [5]:
sys.path.append("../src")
import emd_loader
import labram
import biot
import cbramod
import classical_features
#import labram_special
import luna

import helpers
from helpers import save_features, load_features

importlib.reload(emd_loader)
importlib.reload(helpers)

<module 'helpers' from '/content/helpers.py'>

# Loading data

In [6]:
from google.colab import drive
drive.mount("/content/drive/", force_remount=True)
emd_dir = Path("/content/drive/MyDrive/eeg_moments_dataset")

Mounted at /content/drive/


In [ ]:
from typing import Callable
# The feature extraction is designed this way, because of random crashes of google collab

sub_ids = range(1,6+1)
ses_ids = range(1,8+1)

def compute_individual_features(
    feature_dir: Path,
    embed: Callable[[np.ndarray, list[str]], np.ndarray]
):
    for sub_id in tqdm(sub_ids, desc="Subjects"):
        for ses_id in tqdm(ses_ids, leave=False, desc="Sessions"):
            feature_path = feature_dir / f"feature-{sub_id}-{ses_id}.h5"
            if feature_path.exists():
                continue
            dataset = emd_loader.EmdDataset(emd_dir)
            tmp_X, ch_names, tmp_Y = dataset.get_data_by_ids(sub_id, ses_id)
            res = embed(tmp_X, ch_names)
            save_features(feature_path, res, tmp_Y)

def merge_features(feature_dir: Path):
    all_features_path = feature_dir / f"features.h5"
    if all_features_path.exists():
        return
    X = []
    Y = []
    for sub_id in tqdm(sub_ids, desc="Subjects"):
        for ses_id in tqdm(ses_ids, leave=False, desc="Sessions"):
            feature_path = feature_dir / f"feature-{sub_id}-{ses_id}.h5"
            tmp_X, tmp_Y = load_features(feature_path)
            X.append(tmp_X)
            Y.append(tmp_Y)
    X = np.concatenate(X)
    Y = np.concatenate(Y)
    save_features(all_features_path, X, Y)

# All input to embeddings is given in format (n_trials, n_channels, n_times)
# with unit micro Volt
embeddings = [
    (
        Path("/content/drive/MyDrive/eeg_features/emd_biot"),
        lambda X, ch_names: biot.biot_embed(X, 500, ch_names, batch_size = 32, device = "cuda"),
    ),
    (
        Path("/content/drive/MyDrive/eeg_features/emd_luna"),
        lambda X, ch_names: luna.luna_embed(X, 500, ch_names, batch_size = 32, device = "cuda"),
    ),
    (
        Path("/content/drive/MyDrive/eeg_features/emd_labram"),
        lambda X, ch_names: labram.labram_embed(X, 500, ch_names, batch_size = 32, device = "cuda"),
    ),
    (
        Path("/content/drive/MyDrive/eeg_features/emd_cbramod"),
        lambda X, _: cbramod.cbramod_embed(X, 500, batch_size = 32, device = "cuda"),
    ),
]

#feature_dir = Path("/content/drive/MyDrive/eeg_features/emd_labram_special")

for dir, func in embeddings:
    print(f"Start {dir}")
    dir.mkdir(exist_ok=True, parents=True)

    compute_individual_features(dir, func)
    merge_features(dir)

Start /content/drive/MyDrive/eeg_features/emd_labram


Sessions:   0%|          | 0/8 [00:00<?, ?it/s]
                                               
Sessions:   0%|          | 0/8 [00:00<?, ?it/s]
                                               
Sessions:   0%|          | 0/8 [00:00<?, ?it/s]
                                               
Sessions:   0%|          | 0/8 [00:00<?, ?it/s]
                                               
Sessions:   0%|          | 0/8 [00:00<?, ?it/s]
                                               
Sessions:   0%|          | 0/8 [00:00<?, ?it/s]
                                               
Sessions:   0%|          | 0/8 [00:00<?, ?it/s]
                                               
Sessions:   0%|          | 0/8 [00:00<?, ?it/s]
                                               
Subjects: 100%|██████████| 6/6 [00:00<00:00, 20.73it/s]


Start /content/drive/MyDrive/eeg_features/emd_biot


Sessions:   0%|          | 0/8 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/158 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 13.8MB            

model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/torch/functional.py:681: UserWarning: A window was not provided. A rectangular window will be applied,which is known to cause spectral leakage. Other windows such as torch.hann_window or torch.hamming_window are recommended to reduce spectral leakage.To suppress this warning and use a rectangular window, explicitly set `window=torch.ones(n_fft, device=<device>)`. (Triggered internally at /pytorch/aten/src/ATen/native/SpectralOps.cpp:835.)
  return _VF.stft(  # type: ignore[attr-defined]

Sessions:  38%|███▊      | 3/8 [01:28<02:30, 30.14s/it]